In [2]:
# ── Colab setup (run this cell first!) ─────────────────────────────────────
import os, urllib.request

# Tell Keras to use JAX as its computing backend instead of TensorFlow.
# JAX should install cleanly on all platforms, including Apple Silicon Macs. (Fingers crossed)
os.environ.setdefault("KERAS_BACKEND", "jax")

# Enable interactive widgets (sliders, buttons) inside Google Colab.
# (and silently skip when running locally)
try:
    from google.colab import output as _out
    _out.enable_custom_widget_manager()
except ImportError:
    pass

# Download the utils and assignment files from GitHub into this Colab session
_RAW = "https://raw.githubusercontent.com/franzkeaw/NMRC-practicals-2026/main/"
for _f in ['utils.py', 'utils_assignment_1.py']:
    if not os.path.exists(_f):  # skip if already downloaded in this session
        urllib.request.urlretrieve(_RAW + _f, _f)
# ─────────────────────────────────────────────────────────────────────────────

# Modeling a single neuron

In this first tutorial, we are going to explore a very simple computational model for the neuron. In spite of its simplicity, this model is widely used in artificial neural networks. In particular, we are going to explore the characteristics of different activation functions. 

### Question 1
In the lecture, you saw the following equation:

\begin{equation}
a_i = f \left( \sum_{j=1}^N w_{ij}a_j-b_i \right)
\end{equation}

Which aspects of the biological neuron are being modeled by $a_j$, $w_{ij}$, $b_i$, $f$, and $a_i$? Write your answer in the cell below.

### Answer 1
$a_j$: Activity of the presynaptic neuron j, in firing rate.<br>
$w_{ij}$: Weight, so the strength of the synapses j to neuron i <br>
$b_{i}$: Bias, a value which determines how quickly a neuron will respond to particular inputs.<br>
$f$: Transfer function, so the transduction of the incoming input to the output of the neuron. <br>
$a_i$: Output of the neuron i, in firing rate.<br>


[**Background**: This is the classic “rate model” of a neuron, the same abstraction used in many artificial neural network types. The idea is to describe a neuron not by individual spikes but by its average firing rate, a number between 0 and (roughly) some maximum. The presynaptic activities $a_j$ are combined linearly: each presynaptic neuron “votes” with a strength given by its synapse $w_{ij}$, positive for excitatory synapses and negative for inhibitory ones. The bias $b_i$ essentially plays the role of the neuron’s intrinsic excitability (think of it as a threshold that needs to be "overcome". The transfer function $f$ then squashes this total input into the output firing rate.]

### Question 2
Which physiological aspects of real (biological) neurons are missing in this model?

### Answer 2
Several aspects of real neurons are missing in this model, because this model is a gross oversimplification. 
Some examples: <br>

Timing of spikes, there is no time component in this model.<br>
Abundancy of dendrites / axon branches and their spatial arrangement.<br>
Refractory period.<br>
Adaptation, plasticity.<br>
No biomolecular effects such as aging, hormones, disease etc.<br>
A biologically plausible transfer function; a hardlimit transfer function is not possible in real life.<br>
Effects of LTP.<br>

[**Background**: Biological neurons are dynamical, spatially extended, adaptive devices. A real neuron integrates thousands of synaptic inputs across its dendritic tree, many of them nonlinearly, and emits discrete action potentials whose precise timing carries information. After firing, it is briefly refractory, which shapes its firing statistics. On longer time scales, synapses change with use (plasticity), and the neuron’s own excitability adapts. The rate model (shown here) deliberately throws all of this away and keeps only “average input --> average output”. Be aware that this is a modelling choice: it captures the computational essence (nonlinear input–output transformation) while remaining analytically and computationally tractable.]

We consider a neuron, which receives input from a single presynaptic neuron. We will now drop the subscripts and call the output of the neuron of interest $a$, the output of the presynaptic neuron $p$, and the bias $b$.

We are interested in understanding how the output of our neuron behaves as a function of the input. We begin by writing up a short function that, for a given weight $w$, input $p$, and bias $b$,  and for a given transfer function, will compute the output of our neuron and plot it. In the cell below is a chunk of Python code, where I import the function `plot_single_neuron_output`, which we will be using throughout the tutorial. Make sure you run the cell below before continuing, so that the function becomes available when we want to call it later on in the notebook. 

In [3]:
from utils_assignment_1 import plot_single_neuron_output
from ipywidgets import interact, fixed

ModuleNotFoundError: No module named 'numpy'

Let's call the function in an interactive way, so we can nicely visualize the effect of changing parameters. For now we consider two settings, namely $w=1$ and $w=-1$, and we keep $b$ fixed at $b=0.5$. We choose the hard-limit function (`hardlim`) to start with. Run the cell below: you will see two sliders, with which you can set $w$ to either 1 or $-1$ and vary $p$ between 0 and 1.

In [2]:
interact(plot_single_neuron_output, w=(-1, 1, 2), p=(0.0, 1.0, 0.05), b=fixed(0.5), tf_name=fixed('hardlim'));

### Question 3
Describe how output $a$ (the horizontal red line) varies as a function of $p$ (where $p$ is between 0 and 1), for both cases $w=1$ and $w=-1$.

### Answer 3
$w=1$: This neuron simply multiplies the input $p$ by 1, thus doesn't change it. If this reached threshold $b = 0.5$ it fires, i.e. the output $a = 1$<br>
$w=-1$: This neuron received negative input from the negative $w$, so this can be seen as inhibitory input. With $p$ between 0 and 1, it will never reach threshold and spike (i.e. never reach $a = 1$)

[**Background**: With a hard-limit transfer function the neuron is a threshold device: it compares the weighted input against the bias and answers “yes” (1) or “no” (0). The bias sets where the threshold sits on the input axis: e.g., here at $p = 0.5$. The sign of the weight decides which side of the threshold is “yes”. A negative weight flips the neuron’s preference: it now responds to small inputs rather than large ones. In the plot, the red output line jumps between 0 and 1 exactly where the (dashed) input line crosses the threshold; everywhere else it is flat. This all-or-none behaviour is the defining feature of a classifier-type neuron, and it is what makes a single neuron equivalent to a logic gate (the topic of the next practical).]

Assume now that the input gets a larger dynamic range and can vary between -4 and 4. Run the cell below and use the slider to set $w$ and vary the value of $p$ in this range.

In [4]:
interact(plot_single_neuron_output, w=(-1, 1, 2), p=(-4.0, 4.0, 0.05), b=fixed(0.5), tf_name=fixed('hardlim'));

NameError: name 'interact' is not defined

### Question 4
Describe again how the output $a$ varies as a function of $p$.

### Answer 4
When $p$ is allowed to vary from -4 to 4:<br>
$w=1$: As before, the input $p$ is multiplied by 1, thus stays the same. Ergo, when $p$ reached threshold $b = 0.5$ the cell will spike, as in $a = 1$.<br>
$w=-1$: The input is multiplied by $w = -1$. In this case, a negative input will be able to reach threshold, i.e. $p$ < $-0.5$. Other inputs will not reach threshold.

[**Background**: Widening the input range reveals that the neuron’s response is always a step, where only the position and direction of the step change. With $w=1$ the step is “upwards” (low input --> 0, high input --> 1); with $w=-1$ it is “downwards” (high input --> 0, low input --> 1). Geometrically, $w$ and $b$ together define a decision boundary and the sign of $w$ determines on which side of $p^$ the neuron is active. This is the “line separating two classes of points” idea that the next practical will look into. Note that the boundary always sits at $p^ = b/w$: changing $b$ moves the threshold, changing $w$ moves and possibly flips it.]

We now explore the effect of $b$ on the output. We further assume that we receive a binary signal as input ($p$ is either 0 or 1). Run the cell below, and use the slider to vary the value of $b$. Understand how that affects the transfer function. 

In [4]:
interact(plot_single_neuron_output, w=(-1, 1, 2), p=(0, 1, 1), b=(-2, 2, 0.05), tf_name=fixed('hardlim'));

### Question 5
We again study both cases $w=1$ and $w=-1$. For what ranges of the bias $b$ will our neuron be able to discriminate a binary input (which takes on either 1 or 0) coming from the presynaptic cell?

### Answer 5
$w=1$: Here the best classification between 1 and 0 is the middle, i.e. $b=0.5$. As the weight won't affect the $p$ value, inputs of 0 will be < $b = 0.5$, and inputs of 1 > $b$, giving appropriate outputs. If the input will only be 0 or 1, $b$ should be in the range $0 < b \leq 1$.<br>
$w=-1$: When the input is multiplied by $-1$, it's basically the same but negative, ergo $b$ should be in the range $-1 < b \leq 0$.

[**Background**: With a binary input, “discriminating” means implementing a "non-trivial boolean function" (where non-trivial means that the output actually changes depending on the input): either the identity, where the output matches the input ($w=1$) or the negation ($w=-1$). The bias must sit between the two weighted inputs: for $w=1$ the weighted inputs are $0$ and $1$, so any threshold strictly between them works:  the interval $(0, 1]$ under the >= convention, because the neuron fires when the input reaches the threshold. For $w=-1$ the weighted inputs are $0$ and $-1$, and the neuron must fire for the larger one ($p=0$), giving $(-1, 0]$ ]

Transfer functions can have different ranges (codomain). You can take a look at this [Wikipedia entry](https://en.wikipedia.org/wiki/Activation_function) for a review of the topic and a list of popular functions. Importantly, while the output of the `hardlim` function is between 0 and 1, that need not be the case. Here we introduce two new functions.
\begin{align}
\text{Linear function:}\quad & \texttt{purelin(x)} & \rightarrow & \quad f(x)=x \\
\text{Hyperbolic tangent:} \quad & \texttt{tanh(x)} &\rightarrow & \quad f(x)=\frac{e^{2x}-1}{e^{2x}+1}
\end{align}
Note that the codomain of the linear function is $(-\infty, \infty)$, while for the hyperbolic tangent it is $(-1, 1)$. Now, run the cell below, use the drop down menu to select different transfer functions, and the slider to change the value of the bias. 

In [5]:
interact(plot_single_neuron_output, w=(-1, 1, 2), p=(0, 1, 1), b=(-2, 2, 0.05), tf_name=['hardlim', 'hardlims', 'purelin', 'tanh', 'sigmoid']);

### Question 6
What is the common effect of changing the bias on the output of all the transfer functions?

### Answer 6
It shifts the transfer function along the input axes (in this case the $x$-axis). The shape of the functions remains the same.

[**Background**: In the equation $a = f(wp - b)$, the bias enters inside the argument of $f$, together with the input. Whatever $f$ looks like, adding a constant to its argument just translates its graph along the input axis; it cannot stretch, flip or reshape it. This is why the bias is often described as setting the neuron’s "threshold" or resting point: it decides at which input level the neuron starts to respond, not how it responds. That's the reasons the bias is such a convenient parameter: one scalar (=one single "number") controls the “operating point” of the neuron independently of the gain (controlled by $w$) and the nonlinearity (controlled by $f$).]

### Question 7
Which configuration of transfer function and parameters classifies values of the input $p$ most strictly into 'categories'? We refer to transfer functions that make such a strong distinction between two ranges of input values as _classifiers_. 

### Answer 7
To make a strong classifier, the hardlimit transfer function would be ideal. The values of $b$ and $w$ will depend on what you want to classify, but the $w$ can't be 0. Dependent on your input $p$, your threshold should be of a logical value, i.e. inputs and weight of 1 and a threshold of 500 will not classify. 
<br>
For instance, as seen in previous questions, classifying binary inputs to binary outputs (a really boring classification), can be readily done with $0 < b \leq 1$, and a $w = 1$

[**Background**: A “classifier” transfer function has to decide: given an input, which category does it belong to? The ideal decision function is a step:  0 on one side of the boundary, 1 on the other, with no 'hesitation' in between. hardlim is exactly that ideal, which is why it is often the standard choice when the goal is a categorical decision, and why a neuron with hardlim is literally a logic gate (next practical). Smooth sigmoids such as tanh "approach" this ideal: far from the threshold they saturate at their extremes (e.g. at 0 and 1, remember the slides from the math lecture) and behave almost like a step, but near the threshold they produce intermediate values(!) One could argue that the "price of smoothness is decisiveness", and, as the next questions explore, the benefit is the ability to smoothly grade responses rather than abruptly switch them.]

### Question 8
Conversely, which configuration is the worst at grouping values of the input in separate categories? 

### Answer 8
A linear transfer function (like purelin) will not give strict classifications, but a continuous output.

[**Background**: A linear transfer function passes information through unchanged (up to scaling and shifting): $a = wp - b$. There is no threshold, no saturation, no “categories”: the output is a graded reflection of the input. That makes purelin a terrible classifier but an excellent estimator: if the task is to reproduce or transmit a quantity (e.g. represent the input’s magnitude, or propagate a signal 'faithfully' through a layer), linearity is exactly what you want. This trade-off (sharp categorisation vs. graded representation) recurs throughout the course: E.g. hardlim for decisions and sigmoids for smooth probability-like outputs (between 0 and 1).]

### Question 9
What could be the advantage of a poorly classifying configuration?

### Answer 9
This will be an estimator. If you want your output to reflect the input in a continuous, linear fashion, you would need this configuration.

[**Background**: Classification is only one thing neurons (and networks) are used for. Often we want a system whose output tracks its input, think of e.g. a thermostat reading, a motor command, a predicted value. A linear or gently nonlinear transfer function does this: small input changes produce small output changes, so the output carries the input’s graded structure. There is a second, deeper advantage that this course builds towards: learning. Gradient-based learning (as we will see in a later practical) needs derivatives; a hard threshold has derivative zero almost everywhere, so it cannot be trained by gradient descent. Smooth transfer functions have useful derivatives everywhere, which is why real networks often use sigmoids, tanh or ReLU rather than hard limits, even when the final decision is categorical.]

### Question 10
Which transfer function is the most biological one? Explain!

### Answer 10
Of our transfer functions the hyperbolic tangent (tanh) function is the most biologically plausible one. These shapes of curves are often seen in biology and psychology (for those interested, google psychometric curves).

[**Background**: If you inject a steadily increasing current into a real neuron and record its firing rate, you get a smooth S-shaped curve: silent at low currents, roughly linear in a middle range, and saturating at a maximal rate.  Sigmoidal functions (logistic, tanh,…) are the standard way to capture this f–I (frequency–current) curve. The same S-shape appears in psychology as the “psychometric curve” relating stimulus intensity to detection probability. Biologically, the smoothness comes from the probabilistic, noise-driven nature of spike generation, and the saturation from the finite maximal firing rate set by the refractory period. Modelling $f$ as a sigmoid is therefore more or less the closest match to what real neurons actually compute (and it's mathematically convenient, as I already hinted at).]

### Question 11
Consider the transfer function that you gave as an answer in the previous question. Go back to the last plot, and set it up using $w=1$ and $b=0$. In which input range does the transfer function behave as an estimator of the input, and in which range as a classifier?

### Answer 11
Estimator range: The middle of the tansig curve is relatively straight, and acts as an estimator, much like a linear transfer function.<br>
Classifier range: The ends of the tansig curve plateau towards -1 and +1, and this acts similar to the hard limit transfer function.

[**Background**: A single sigmoidal neuron is a hybrid device, and which “mode” it is in depends on where on the curve the inputs live. Near the origin the S-curve is close to a straight line, so the neuron estimates: output ≈ input, and graded information is preserved. Far from the origin the curve flattens into its two plateaus, so the neuron classifies: any sufficiently negative input gives ≈ −1 (or zero, depending on the exact function), any sufficiently positive input gives ≈ +1, and all the fine structure of the input is discarded. The transition between the two regimes is gradual: this is the fundamental difference from hardlim, which is a pure classifier everywhere. Real neurons operate in both regimes: a weak, fluctuating input around rest is represented in a graded fashion, while strong, decisive inputs elicit an effectively binary (high-rate vs. silent) response.]

### Question 12
What is the essential mathematical feature of a transfer function aimed at classification?

### Answer 12
The feature of non-lineararity; it needs a relatively strict distinction between values, such as we see in the hard limit function. This is highly non-linear.

[**Background**: What makes a function a classifier is that it compresses a continuum of inputs into (essentially) two outputs. Mathematically this requires the function to be far from linear: a linear map is one-to-one and preserves all distinctions between inputs, so it can never merge “similar” inputs into one category. A saturating nonlinearity does the merging: everything below the threshold collapses to one output level, everything above to the other. The steepness of the transition controls how “strict” the classification is (how much input change is tolerated within a category), and the boundedness guarantees the output really is categorical.]

### Question 13
Which aspects of how neurons render the transfer from synaptic input to output can be modeled with a sigmoid function?

### Answer 13
The sigmoid function has a threshold-like point, it has to reach certain input to start responding. Moreover, in the middle, a small increase in input gives a large increase in output (i.e. it has a dynamic range). furthermore, the function also has a maximum, where the output doesn't increase further.

[**Background**: The sigmoid $\sigma(x) = 1/(1+e^{-x})$ is the workhorse activation function of neural network history. Its shape mirrors the empirical f–I curve of real neurons: flat near zero output for weak inputs, steep in the middle, flat again near the maximum. Several properties make it especially useful: First, it is smooth and differentiable everywhere, with the convenient derivative $\sigma’(x) = \sigma(x)(1-\sigma(x))$ (the key property that makes backpropagation work, as we will see later). Second, its output is bounded in (0,1), which can be read as a probability or a normalised firing rate.]

### Question 14
Which physiological aspects of real neurons can be seen as the counterpart of the bias in computational neurons?

### Answer 14
This can be thought of as the resting potential of a neurons, spontaneous input of other neurons on said neuron, biomolecular effects (i.e. hormones etc).

[**Background**: In a real neuron, whether it fires is decided by the total synaptic current plus the cell’s own state. The resting membrane potential sits around (roughly) −70 mV, below the firing threshold of (roughly) −55 mV (although this varies between neuron types and conditions); anything that depolarises the cell effectively reduces the distance to threshold. That distance is exactly what the bias parameterises in the model: with the sign convention of this notebook ($a = f(\sum w_j a_j - b)$), a larger $b$ means the neuron needs more input to fire. Different neuron types differ systematically in these intrinsic properties: this is a major reason real networks are heterogeneous. Conceptually, the bias also plays a mathematical role: without it, many classification problems (e.g. AND with inputs in {0,1}) would become unsolvable by a single neuron, as we will see in the next practical.]